In [ ]:
# Uncomment these lines if running on Google Colab
import os
os.environ["KAGGLEHUB_CACHE"] = "/content/data"

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np

class UnderwaterImageryDataset(Dataset):
    def __init__(self, root_dir, csv_file, transform=None, target_transform=None):
        self.root_dir = root_dir
        self.metadata = pd.read_csv(csv_file)
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir,"dataset", "Image", self.metadata.iloc[idx, 0])
        mask_path = os.path.join(self.root_dir,"dataset", "Mask", self.metadata.iloc[idx, 1])

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # Convert mask to grayscale (1 channel = binary segmentation)

        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            mask = self.target_transform(mask)

        mask = remap_mask(mask)

        return image, mask

In [ ]:
class UnderwaterImageryDataset(Dataset):
  def __init__(self, image_paths, mask_paths, transform=None, target_transform=None):
    self.image_paths = image_paths
    self.mask_paths = mask_paths
    self.transform = transform
    self.target_transform = target_transform

  def __len__(self):

    return len(self.image_paths)

  def __getitem__(self, idx):
    image = Image.open(self.image_paths[idx]).convert("RGB")
    mask = Image.open(self.mask_paths[idx]).convert("L")

    if self.transform:
      image = self.transform(image)

    if self.target_transform:
      mask = self.target_transform(mask)
      mask = remap_mask(mask)
    return image, mask

In [ ]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# Image transforms (Resize, ToTensor, Normalize with ImageNet stats)
image_transforms = transforms.Compose([
  transforms.ToTensor(),
  transforms.Resize((256, 256)),
  transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Mask transforms (Resize, PILToTensor)
mask_transforms = transforms.Compose([
  transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),
  transforms.PILToTensor(),
])

In [ ]:
image_paths = os.path.join(path, "dataset", "images")
mask_paths = os.path.join(path, "dataset", "masks")

In [ ]:
# Split into train and test sets (80% train, 20% test)

# YOUR CODE HERE
train_images, test_images, train_masks, test_masks = train_test_split(
  image_paths, mask_paths, test_size=0.3, random_state=42
)
# Create Dataset objects
train_dataset = UnderwaterImageryDataset(train_images, train_masks, transform=image_transforms, target_transform=mask_transforms)
test_dataset = UnderwaterImageryDataset(test_images, test_masks, transform=image_transforms, target_transform=mask_transforms)

# Create DataLoaders
BATCH_SIZE = 16

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
train_dataset = UnderwaterImageryDataset(
    root_dir=os.path.join(path, "train"),
    transform=image_transforms
)


val_dataset = UnderwaterImageryDataset(
    root_dir=os.path.join(path, "validation"),
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train Dataset: {len(train_dataset)} images")
print(f"Test Dataset: {len(test_dataset)} images")

In [ ]:
import matplotlib.pyplot as plt

# Display some images with their masks
for i in range(10,13):
    img, mask = train_dataset[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img.permute(1, 2, 0))  # Convert (C, H, W) to (H, W, C)
    axes[0].set_title("Image")
    axes[0].axis("off")
    axes[1].imshow(mask.squeeze(), cmap="gray")
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")
    plt.show()


In [ ]:
# TO DO
import segmentation_models_pytorch as smp

device = "cpu"
model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
).to(device)

In [ ]:
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)  # mask shape becomes [N, H, W]

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)    # mask shape becomes [N, H, W]

            outputs = model(images)  # Now [N, H, W]
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# TO DO
import torch
from torch import nn
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 10  # Define number of epochs
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")


In [ ]:
# TO DO
import matplotlib.pyplot as plt

plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()